# 第73章 Olist电商物流履约分析

使用 Olist 巴西电商近 10 万笔真实公开订单，连接订单与商品明细，诊断承运时效、延期风险、运费负担和卖家履约表现。

## 项目背景

Olist 公开数据覆盖巴西多卖家电商平台 2016-2018 年订单。运营团队需要判断延期发生在出库还是运输环节、哪些卖家和月份风险更高，以及运费对商品价值的负担。数据来自 Olist 发布的 Brazilian E-Commerce Public Dataset；客户评价并不能证明某个物流环节导致满意度变化。

## 学习目标

- 建立订单级物流宽表
- 正确处理未签收与时间缺失
- 衡量采购到发货、在途和总履约时长
- 识别延期订单的时间与卖家集中度
- 把诊断转化为物流 SLA 与监控建议


## 数据字典

| 字段 | 含义 | 使用说明 |
| --- | --- | --- |
| order_id | 订单编号 | 订单表与明细表连接键 |
| order_status | 订单状态 | delivered/canceled 等 |
| order_purchase_timestamp | 下单时间 | 流程起点 |
| order_delivered_carrier_date | 交承运商时间 | 出库完成 |
| order_delivered_customer_date | 客户签收时间 | 实际完成 |
| order_estimated_delivery_date | 预计送达时间 | 承诺基线 |
| price/freight_value | 商品价/运费 | 订单明细金额 |
| seller_id | 卖家编号 | 履约责任维度 |

## 数据质量检查清单

- 订单主键是否唯一
- 订单明细一对多连接后是否重复计算订单
- 各里程碑时间缺失与逆序
- 未签收订单不能计算实际履约时长
- 异常负时长和极端长尾
- 金额字段是否非负


## 项目任务

1. 读取并连接订单与明细
2. 聚合到订单粒度避免重复
3. 构造出库、运输、总履约与延期指标
4. 分析月度 SLA
5. 定位卖家延期集中度
6. 输出物流运营行动清单


## 项目交付物

- 一份可复现的分析 Notebook
- 清洗规则与关键指标表
- 至少一张支持结论的图表
- 结论、限制和下一步建议

## 阶段检查点

- [ ] 问题和数据字典完成
- [ ] 质量检查和清洗记录完成
- [ ] 核心指标或图表完成
- [ ] 结论与限制完成

## 最低完成标准

- 每个代码阶段都有可见输出，不能依赖未展示的隐藏状态。
- 所有关键清洗、筛选和评价口径都写在 Markdown 或注释中。
- 最终结论至少引用一个数值或图表证据，并说明适用范围。

## 提升任务

完成基础验收后，可以增加一个对照方案、一个分组切片或一个参数敏感性实验，比较结果是否稳定。


## 1. 读取订单与商品明细

订单表是一单一行，商品表是一单多行；先分别审计，再聚合连接。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

date_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"]
orders = pd.read_csv(f"{base_url}/datasets/olist_orders_dataset.csv", parse_dates=date_cols)
items = pd.read_csv(f"{base_url}/datasets/olist_order_items_dataset.csv", parse_dates=["shipping_limit_date"])
print("订单/明细:", orders.shape, items.shape)
print("订单主键重复:", orders.order_id.duplicated().sum(), " 无明细订单:", (~orders.order_id.isin(items.order_id)).sum())
print("状态分布:\n", orders.order_status.value_counts())


## 2. 构造订单级物流宽表

先把商品价、运费和卖家数聚合到订单级，避免多商品订单把时效重复计权。


In [ ]:
item_order = items.groupby("order_id").agg(item_count=("order_item_id", "size"), goods_value=("price", "sum"), freight_value=("freight_value", "sum"), seller_count=("seller_id", "nunique"))
seller_order = items.groupby("order_id").seller_id.first().rename("primary_seller")
logistics = orders.merge(item_order, on="order_id", how="left").merge(seller_order, on="order_id", how="left", validate="one_to_one")
logistics["dispatch_days"]=(logistics.order_delivered_carrier_date-logistics.order_purchase_timestamp).dt.total_seconds()/86400
logistics["transit_days"]=(logistics.order_delivered_customer_date-logistics.order_delivered_carrier_date).dt.total_seconds()/86400
logistics["lead_days"]=(logistics.order_delivered_customer_date-logistics.order_purchase_timestamp).dt.total_seconds()/86400
logistics["delay_days"]=(logistics.order_delivered_customer_date-logistics.order_estimated_delivery_date).dt.total_seconds()/86400
logistics["late"]=logistics.delay_days.gt(0)
logistics["freight_ratio"]=logistics.freight_value/logistics.goods_value.replace(0, np.nan)
print(logistics[["dispatch_days", "transit_days", "lead_days", "delay_days", "freight_ratio"]].describe(percentiles=[.5,.9,.95,.99]).round(2))


## 3. SLA与流程瓶颈

只在已签收订单上衡量实际时效；分别看出库和运输才能定位责任环节。


In [ ]:
delivered=logistics[(logistics.order_status=="delivered")&logistics.order_delivered_customer_date.notna()].copy()
valid=(delivered.dispatch_days>=0)&(delivered.transit_days>=0)&(delivered.lead_days>=0)
delivered = delivered.loc[valid]
delivered["purchase_month"]=delivered.order_purchase_timestamp.dt.to_period("M").astype(str)
monthly = delivered.groupby("purchase_month").agg(orders=("order_id", "size"), late_rate=("late", "mean"), median_dispatch=("dispatch_days", "median"), median_transit=("transit_days", "median"), p90_lead=("lead_days", lambda x:x.quantile(.9)))
print("已签收有效订单:", len(delivered), " 延期率:", f"{delivered.late.mean():.1%}")
print(monthly.tail(12).round(2))
monthly.late_rate.plot(figsize=(10,4), marker="o", title="按下单月的延期率"); plt.ylabel("延期率"); plt.tight_layout(); plt.show()


## 4. 卖家与运费风险

设置最小订单量，避免用少量订单给卖家贴标签。高延期率只是筛查信号，还需拆分承运商和地区。


In [ ]:
seller = delivered.groupby("primary_seller").agg(orders=("order_id", "size"), late_rate=("late", "mean"), median_dispatch=("dispatch_days", "median"), median_freight_ratio=("freight_ratio", "median"))
eligible=seller.query("orders>=30").sort_values(["late_rate", "orders"], ascending=False)
print("订单>=30的高延期卖家:\n", eligible.head(12).round(3))
print("运费占商品价值中位数:", f"{delivered.freight_ratio.median():.1%}", " P90:", f"{delivered.freight_ratio.quantile(.9):.1%}")


## 5. 物流运营结论

形成可执行的 SLA 分层、监控和后续数据需求。


In [ ]:
worst = monthly.late_rate.idxmax(); p90=delivered.lead_days.quantile(.9)
print(f"1. 月度延期峰值出现在 {worst}，回查当月卖家出库与承运时长。")
print(f"2. 总履约 P90 为 {p90:.1f} 天，建议按地区/品类进一步建立差异化承诺。")
print(f"3. 将 {len(eligible)} 个订单量>=30的卖家纳入分层SLA看板，重点看延期率与出库中位数。")
print("限制：公开数据不含承运商、仓库节点和实时轨迹；卖家差异可能受地区与品类结构影响，不能直接归因。")


## 结论与表达

- 物流时效必须在订单粒度计算。
- 延期要拆成出库和在途环节。
- 卖家排名必须设置样本量门槛并做结构校正。
- 公开订单数据适合诊断，不足以做承运商因果评估。


## 项目验收清单

- 连接后保持一单一行
- 能解释未签收订单的处理
- 能计算月度延期率和 P90
- 建议包含 SLA、责任环节和数据限制

建议重新启动内核后从第一个代码单元格运行，确认项目不依赖隐藏状态。


## 本章小结

使用 Olist 巴西电商近 10 万笔真实公开订单，连接订单与商品明细，诊断承运时效、延期风险、运费负担和卖家履约表现。


### 你已经完成

- 建立订单级物流宽表
- 正确处理未签收与时间缺失
- 衡量采购到发货、在途和总履约时长
- 识别延期订单的时间与卖家集中度
- 把诊断转化为物流 SLA 与监控建议


### 项目流程速查

| 阶段 | 交付内容 |
| --- | --- |
| 步骤 1 | 读取并连接订单与明细 |
| 步骤 2 | 聚合到订单粒度避免重复 |
| 步骤 3 | 构造出库、运输、总履约与延期指标 |
| 步骤 4 | 分析月度 SLA |
| 步骤 5 | 定位卖家延期集中度 |
| 步骤 6 | 输出物流运营行动清单 |


### 质量与结论提醒

- 订单主键是否唯一
- 订单明细一对多连接后是否重复计算订单
- 各里程碑时间缺失与逆序
- 物流时效必须在订单粒度计算。
- 延期要拆成出库和在途环节。
- 卖家排名必须设置样本量门槛并做结构校正。
- 公开订单数据适合诊断，不足以做承运商因果评估。


### 项目交付检查

- [ ] 连接后保持一单一行
- [ ] 能解释未签收订单的处理
- [ ] 能计算月度延期率和 P90
- [ ] 建议包含 SLA、责任环节和数据限制


### 后续迭代建议

完成验收后，记录一个最值得继续验证的假设：可以是更多数据、不同时间窗口、另一种模型，或一个更细的分组分析。
